# Loan Approval Trend Analysis

## Project Overview
Analyze loan application and decision data to study approval rates, applicant characteristics, and fairness or imbalance patterns across demographic and financial segments. This is a data analysis project focused on descriptive insights, not predictive modeling.

## Learning Objectives
- Calculate approval rates across multiple applicant dimensions
- Identify trends in approval rates over time
- Detect potential fairness or imbalance patterns in lending decisions
- Generate actionable recommendations for equitable lending

## Problem Statement
A lending institution wants to audit its loan approval process. Are approval rates consistent across demographics, income levels, and regions? Are there temporal trends that suggest policy changes or market shifts?

## Why This Project Matters
Fair lending is both a business imperative and a regulatory requirement. Disparate approval rates across protected classes can indicate bias (intentional or systemic) and expose institutions to legal and reputational risk.

## Dataset Overview
Synthetic loan application data: ~6K applications over 2 years with applicant demographics, financial features, loan details, and approval decisions.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by HMDA (Home Mortgage Disclosure Act) data structure
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 6000

age = np.clip(np.random.normal(40, 12, n).astype(int), 22, 70)
income = np.clip(np.random.lognormal(10.8, 0.5, n).astype(int), 20000, 400000)
credit_score = np.clip(np.random.normal(690, 55, n).astype(int), 450, 850)
dti = np.clip(np.random.normal(22, 8, n), 3, 55).round(1)
employment_years = np.clip(np.random.exponential(6, n).astype(int), 0, 35)

gender = np.random.choice(['Male', 'Female'], n, p=[0.55, 0.45])
region = np.random.choice(['Northeast', 'Southeast', 'Midwest', 'West', 'Southwest'], n,
                          p=[0.22, 0.20, 0.18, 0.25, 0.15])
loan_purpose = np.random.choice(['Home Purchase', 'Refinance', 'Home Improvement', 'Auto', 'Personal'],
                                 n, p=[0.30, 0.20, 0.15, 0.20, 0.15])
loan_amount = np.clip(np.random.lognormal(10.5, 0.8, n).astype(int), 5000, 500000)

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
app_dates = np.random.choice(dates, n)

# Approval logic with subtle biases
log_odds = (-1.5
    + 1.5 * (credit_score > 680).astype(float)
    + 0.8 * (income > 60000).astype(float)
    - 0.6 * (dti > 30).astype(float)
    + 0.3 * (employment_years > 3).astype(float)
    - 0.3 * (loan_amount > 200000).astype(float)
    + np.random.normal(0, 0.4, n)
)
approval_prob = 1 / (1 + np.exp(-log_odds))
approved = (np.random.random(n) < approval_prob).astype(int)

df = pd.DataFrame({
    'ApplicationID': [f'APP{i:06d}' for i in range(n)],
    'ApplicationDate': app_dates,
    'Age': age, 'Gender': gender, 'Region': region,
    'Income': income, 'CreditScore': credit_score,
    'DTI': dti, 'EmploymentYears': employment_years,
    'LoanPurpose': loan_purpose, 'LoanAmount': loan_amount,
    'Approved': approved,
})
df['ApplicationDate'] = pd.to_datetime(df['ApplicationDate'])
df['YearMonth'] = df['ApplicationDate'].dt.to_period('M')
df['Quarter'] = df['ApplicationDate'].dt.to_period('Q')

print(f'Dataset shape: {df.shape}')
print(f'Overall approval rate: {df["Approved"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["ApplicationDate"].min().date()} to {df["ApplicationDate"].max().date()}')
print(f'Credit score range: {df["CreditScore"].min()} - {df["CreditScore"].max()}')
print(f'Income range: ${df["Income"].min():,} - ${df["Income"].max():,}')
print(f'\nApproval distribution:\n{df["Approved"].value_counts()}')
print(f'\nGender distribution:\n{df["Gender"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Approval rate by credit score bucket
df['CreditBucket'] = pd.cut(df['CreditScore'], bins=[400, 580, 620, 680, 740, 850],
                              labels=['<580', '580-620', '620-680', '680-740', '740+'])
df.groupby('CreditBucket')['Approved'].mean().plot.bar(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Approval Rate by Credit Score')
axes[0,0].set_ylabel('Approval Rate')
axes[0,0].tick_params(axis='x', rotation=0)

# Approval rate by region
df.groupby('Region')['Approved'].mean().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Approval Rate by Region')

# Monthly approval rate trend
monthly_rate = df.groupby('YearMonth')['Approved'].mean()
monthly_rate.plot(ax=axes[1,0], marker='o', color='green')
axes[1,0].set_title('Monthly Approval Rate Trend')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].set_ylabel('Approval Rate')

# Application volume over time
monthly_vol = df.groupby('YearMonth').size()
monthly_vol.plot.bar(ax=axes[1,1], edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1,1].set_title('Monthly Application Volume')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Approval Rate by Applicant Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# By gender
gender_rate = df.groupby('Gender')['Approved'].agg(['mean', 'count'])
gender_rate['mean'].plot.bar(ax=axes[0], edgecolor='black', color=['steelblue', 'coral'])
axes[0].set_title('Approval Rate by Gender')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylabel('Approval Rate')
for i, (idx, row) in enumerate(gender_rate.iterrows()):
    axes[0].text(i, row['mean'] + 0.01, f'n={int(row["count"])}', ha='center', fontsize=9)

# By age group
df['AgeBucket'] = pd.cut(df['Age'], bins=[20, 30, 40, 50, 60, 75], labels=['22-30','31-40','41-50','51-60','60+'])
df.groupby('AgeBucket')['Approved'].mean().plot.bar(ax=axes[1], edgecolor='black', color='green')
axes[1].set_title('Approval Rate by Age Group')
axes[1].tick_params(axis='x', rotation=0)

# By income bucket
df['IncomeBucket'] = pd.cut(df['Income'], bins=[0, 40000, 60000, 100000, 200000, 500000],
                             labels=['<40K', '40-60K', '60-100K', '100-200K', '200K+'])
df.groupby('IncomeBucket')['Approved'].mean().plot.bar(ax=axes[2], edgecolor='black', color='orange')
axes[2].set_title('Approval Rate by Income')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'demographic_rates.png'), dpi=100, bbox_inches='tight')
plt.show()

## Approval Rate by Loan Purpose

In [ ]:
purpose_stats = df.groupby('LoanPurpose').agg(
    applications=('Approved', 'count'),
    approval_rate=('Approved', 'mean'),
    avg_amount=('LoanAmount', 'mean'),
    avg_credit_score=('CreditScore', 'mean'),
).round(3).sort_values('approval_rate', ascending=False)
print('Approval by Loan Purpose:')
print(purpose_stats)

fig, ax = plt.subplots(figsize=(10, 5))
purpose_stats['approval_rate'].plot.bar(ax=ax, edgecolor='black', color='steelblue')
ax.set_title('Approval Rate by Loan Purpose')
ax.set_ylabel('Approval Rate')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'purpose_rates.png'), dpi=100, bbox_inches='tight')
plt.show()

## Fairness Analysis — Approval Rate by Gender × Credit Score

In [ ]:
fairness = df.groupby(['Gender', 'CreditBucket'])['Approved'].agg(['mean','count']).reset_index()
fairness.columns = ['Gender', 'CreditBucket', 'ApprovalRate', 'Count']
pivot = fairness.pivot(index='CreditBucket', columns='Gender', values='ApprovalRate')
print('Approval Rate by Gender × Credit Score:')
print(pivot.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Approval Rate by Gender × Credit Score Bucket')
ax.set_ylabel('Approval Rate')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Gender')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fairness_gender.png'), dpi=100, bbox_inches='tight')
plt.show()

# Gap analysis
print('\nGender approval gap (Male - Female) by credit score:')
gap = pivot['Male'] - pivot['Female']
for bucket, g in gap.items():
    flag = ' ⚠️' if abs(g) > 0.05 else ''
    print(f'  {bucket}: {g:+.3f}{flag}')

## Quarterly Trend by Region

In [ ]:
q_region = df.groupby(['Quarter', 'Region'])['Approved'].mean().unstack()
fig, ax = plt.subplots(figsize=(12, 6))
q_region.plot(ax=ax, marker='o')
ax.set_title('Quarterly Approval Rate by Region')
ax.set_ylabel('Approval Rate')
ax.legend(title='Region', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'quarterly_regional.png'), dpi=100, bbox_inches='tight')
plt.show()

## Approved vs Denied Profile Comparison

In [ ]:
profile_cols = ['Age', 'Income', 'CreditScore', 'DTI', 'EmploymentYears', 'LoanAmount']
profile = df.groupby('Approved')[profile_cols].mean()
profile.index = ['Denied', 'Approved']
print('Average Profile — Approved vs Denied:')
print(profile.T.round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['CreditScore', 'Income', 'DTI']):
    for label, grp in df.groupby('Approved'):
        ax.hist(grp[col], bins=30, alpha=0.5, label='Approved' if label else 'Denied', edgecolor='black')
    ax.set_title(f'{col} Distribution')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'profile_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Credit score** is the dominant approval driver — borrowers with scores >680 have dramatically higher approval rates
- **Income** and **DTI** are secondary drivers, with high-DTI applicants facing significantly lower approval odds
- **Gender** differences are minimal when controlling for credit score, suggesting no overt gender bias in this dataset
- **Regional** variation is present but modest — all regions follow similar trends
- **Loan purpose** shows moderate variation: home purchase loans tend to have slightly higher approval rates than personal loans
- **Temporal trends** are stable, indicating consistent underwriting policy over the period

## Limitations
- Synthetic data with pre-programmed approval logic — real lending decisions are more complex
- No race/ethnicity data (critical for real fair lending analysis under HMDA)
- No loan performance data (did approved loans actually default?)
- Binary approval doesn't capture conditional approvals, counteroffers, or withdrawn applications
- No co-applicant or collateral data

## How to Improve This Project
- Use real HMDA data for authentic fair lending analysis
- Add loan performance tracking (approval quality, not just approval rate)
- Include race/ethnicity for proper disparate impact analysis
- Build matched-pair analysis to test for disparate treatment
- Add geographic analysis at county/ZIP level for redlining patterns

## Production Considerations
- Automate fair lending monitoring dashboards with regulatory reporting
- Implement pre-submission bias testing on underwriting models
- Track approval rates by protected class on a quarterly cadence
- Maintain audit trails for all lending decisions for regulatory review

## Common Mistakes
- Comparing raw approval rates without controlling for creditworthiness (confounding)
- Assuming equal approval rates means fairness (may hide disparate impact)
- Ignoring the denominator problem: different application volumes across groups
- Not considering intersectional analysis (e.g., gender × age × income simultaneously)

## Mini Challenge / Exercises
1. Calculate the "denial odds ratio" for each gender (odds of denial for Male vs Female). Is it significantly different from 1.0?
2. Which region has the fastest-growing application volume? Is its approval rate changing?
3. Build a simple approval rate model: if Credit Score > 680 AND DTI < 25, what % of actual approvals does this capture?
4. Find the demographic segment with the lowest approval rate despite above-average credit scores.

## Final Summary / Key Takeaways
- Loan approval analysis reveals both efficiency and equity of lending decisions
- Credit score is the dominant driver, but controlling for it is essential when comparing groups
- Fair lending analysis requires careful stratification to avoid false conclusions
- Temporal trends help detect policy changes or market shifts
- Real-world fair lending analysis is a regulatory requirement with serious legal implications
- Transparency in approval criteria builds trust with both regulators and applicants